In [2]:
import sys
from pathlib import Path

print("Python:", sys.version)
print("Executable:", sys.executable)
print("Working directory:", Path.cwd())

Python: 3.14.0 (tags/v3.14.0:ebf955d, Oct  7 2025, 10:15:03) [MSC v.1944 64 bit (AMD64)]
Executable: D:\ML Challenge\6ab10eb3b23ba_student_resource\student_resource\.venv\Scripts\python.exe
Working directory: D:\ML Challenge\6ab10eb3b23ba_student_resource\student_resource\notebooks


In [3]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print("Project root:", ROOT)
print("src available:", (ROOT / "src").exists())

Project root: D:\ML Challenge\6ab10eb3b23ba_student_resource\student_resource
src available: True


In [4]:
import pandas as pd
import polars as pl
import duckdb
import numpy as np
import scipy
import sklearn
import rapidfuzz
import lightgbm
import pyarrow

print("Pandas:", pd.__version__)
print("Polars:", pl.__version__)
print("DuckDB:", duckdb.__version__)
print("NumPy:", np.__version__)
print("SciPy:", scipy.__version__)
print("Scikit-learn:", sklearn.__version__)
print("RapidFuzz:", rapidfuzz.__version__)
print("LightGBM:", lightgbm.__version__)
print("PyArrow:", pyarrow.__version__)

print("\nALL CORE PACKAGES OK")

Pandas: 3.0.6
Polars: 1.44.2
DuckDB: 1.5.5
NumPy: 2.5.3
SciPy: 1.18.1
Scikit-learn: 1.9.1
RapidFuzz: 3.14.6
LightGBM: 4.7.0
PyArrow: 25.0.1

ALL CORE PACKAGES OK


# Phase 1 — Dataset Audit

Goal: understand the scale, structure, missingness, matching distribution,
and noise characteristics before designing the blocking and ML pipeline.

In [5]:
from pathlib import Path
import os
import sys
import json

import duckdb
import polars as pl
import pandas as pd
import numpy as np

ROOT = Path.cwd().parent
DATASET = ROOT / "dataset"
TRAIN = DATASET / "train"
TEST = DATASET / "test"

print("Project root:", ROOT)
print("Train:", TRAIN)
print("Test:", TEST)

Project root: D:\ML Challenge\6ab10eb3b23ba_student_resource\student_resource
Train: D:\ML Challenge\6ab10eb3b23ba_student_resource\student_resource\dataset\train
Test: D:\ML Challenge\6ab10eb3b23ba_student_resource\student_resource\dataset\test


In [6]:
FILES = {
    "train_source1": TRAIN / "train_source1.tsv",
    "train_source2": TRAIN / "train_source2.tsv",
    "train_source3": TRAIN / "train_source3.tsv",
    "train_ground_truth": TRAIN / "train_ground_truth.tsv",
    "test_source1": TEST / "test_source1.tsv",
    "test_source2": TEST / "test_source2.tsv",
    "test_source3": TEST / "test_source3.tsv",
}

for name, path in FILES.items():
    print(
        f"{name:20} "
        f"exists={path.exists()}  "
        f"size={path.stat().st_size / 1024**2:.2f} MB"
    )

train_source1        exists=True  size=200.34 MB
train_source2        exists=True  size=466.63 MB
train_source3        exists=True  size=480.37 MB
train_ground_truth   exists=True  size=121.13 MB
test_source1         exists=True  size=166.91 MB
test_source2         exists=True  size=485.86 MB
test_source3         exists=True  size=482.56 MB


In [7]:
con = duckdb.connect()

def count_rows(path):
    return con.execute(
        f"""
        SELECT COUNT(*)
        FROM read_csv_auto('{path.as_posix()}', delim='\t')
        """
    ).fetchone()[0]

row_counts = {}

for name, path in FILES.items():
    row_counts[name] = count_rows(path)

row_counts


{'train_source1': 2206821,
 'train_source2': 5034616,
 'train_source3': 5285603,
 'train_ground_truth': 2206821,
 'test_source1': 1732544,
 'test_source2': 4887273,
 'test_source3': 5082316}

In [8]:
row_table = pd.DataFrame(
    [
        {
            "file": name,
            "rows": rows,
            "size_mb": round(FILES[name].stat().st_size / 1024**2, 2)
        }
        for name, rows in row_counts.items()
    ]
)

row_table

,file,rows,size_mb
0,train_source1,2206821,200.34
1,train_source2,5034616,466.63
2,train_source3,5285603,480.37
3,train_ground_truth,2206821,121.13
4,test_source1,1732544,166.91
5,test_source2,4887273,485.86
6,test_source3,5082316,482.56


In [9]:
for name in [
    "train_source1",
    "train_source2",
    "train_source3",
    "test_source1",
    "test_source2",
    "test_source3",
    "train_ground_truth",
]:
    path = FILES[name]

    print("\n" + "=" * 70)
    print(name)
    print("=" * 70)

    schema = con.execute(
        f"""
        DESCRIBE
        SELECT *
        FROM read_csv_auto('{path.as_posix()}', delim='\t')
        """
    ).fetchdf()

    display(schema[["column_name", "column_type"]])


train_source1


,column_name,column_type
0,entity_id,VARCHAR
1,business_name,VARCHAR
2,business_address,VARCHAR
3,country,VARCHAR



train_source2


,column_name,column_type
0,entity_id,VARCHAR
1,business_name,VARCHAR
2,business_address,VARCHAR
3,country,VARCHAR



train_source3


...

,column_name,column_type
0,entity_id,VARCHAR
1,business_name,VARCHAR
2,business_address,VARCHAR
3,country,VARCHAR



test_source3


,column_name,column_type
0,entity_id,VARCHAR
1,business_name,VARCHAR
2,business_address,VARCHAR
3,country,VARCHAR



train_ground_truth


,column_name,column_type
0,source1_entity_id,VARCHAR
1,matched_entity_ids,VARCHAR


In [10]:
for name in [
    "train_source1",
    "train_source2",
    "train_source3",
    "train_ground_truth",
]:
    path = FILES[name]

    print("\n" + "=" * 70)
    print(name)
    print("=" * 70)

    sample = con.execute(
        f"""
        SELECT *
        FROM read_csv_auto(
            '{path.as_posix()}',
            delim='\t'
        )
        LIMIT 5
        """
    ).fetchdf()

    display(sample)


train_source1


,entity_id,business_name,business_address,country
0,S1-925783039,Orelee's Barbershop,"1795 Westchester Drive, High Point, NC",US
1,S1-773889195,Prime Money,"17560 Ellis Road, Tahlequah, OK",US
2,S1-377745466,B+ Retail Inc,"1712 Montebello Avenue, Phoenix, AZ",US
3,S1-133037285,Christ Chapel,"2100 Cameron Drive, Unit APARTMENT G, Dundalk, MD",US
4,S1-755362802,Prabhav Business Center,"797, Lake Town Block A, Kolkata, Howrah, West ...",India



train_source2


,entity_id,business_name,business_address,country
0,S2-166376419,राम मार्केटिंग प्राइवेट लिमिटेड,"KH NO. -570/13, NEW DELHI, WEST DELHI, Delhi",India
1,S2-764573417,-- Holloway Peak Inc Seafood,"105 ELM ST, MORGANTON, NC",US
2,S2-639257739,आदित्य प्रॉपर्टीज एलएलपी,"G-3/571, GULMOHAR COLONY, BHOPAL, Madhya Pradesh",India
3,S2-163963287,Summit Inc,"GREENSBORO, NC, 19 1/2 STARDUST TRAIL",US
4,S2-49942811,Delta Tetlecommunication Inc,"914 PIERPONT AVE, CLEVELAND, OH",US



train_source3


,entity_id,business_name,business_address,country
0,S3-202863386,wilfordhancock.com,"Mack Rd, Haltom City, Texas",US
1,S3-859268022,International South Consultants Private Ltd,NaN,India
2,S3-22467283,LLC Moncada Léarning Center,"5780 Fawn Ct, Fort Worth, Texas",US
3,S3-671162755,Moyna's Coffee,"1 Ivanhoe Ave, PO Box 6009, Cincinnati, Ohio",US
4,S3-960981775,Pvt. EFS Print Ventures Ltd.,"Door No 183, 41St Cross, 22Nd Main 9Th Block J...",India



train_ground_truth


,source1_entity_id,matched_entity_ids
0,S1-965667,"S2-681193310,S2-743505751,S3-775321672,S3-1129..."
1,S1-55344266,"S2-249013014,S2-197070651,S3-478195123,S3-3843..."
2,S1-343815751,"S2-790675320,S2-479876582,S3-878454467"
3,S1-656753428,"S2-153058913,S2-24659151,S3-679606215"
4,S1-102811957,"S2-478959098,S2-553508714,S2-625774905,S3-7280..."


In [11]:
path = FILES["train_ground_truth"]

sample = con.execute(
    f"""
    SELECT *
    FROM read_csv_auto(
        '{path.as_posix()}',
        delim='\t'
    )
    LIMIT 10
    """
).fetchdf()

display(sample)

,source1_entity_id,matched_entity_ids
0,S1-965667,"S2-681193310,S2-743505751,S3-775321672,S3-1129..."
1,S1-55344266,"S2-249013014,S2-197070651,S3-478195123,S3-3843..."
2,S1-343815751,"S2-790675320,S2-479876582,S3-878454467"
3,S1-656753428,"S2-153058913,S2-24659151,S3-679606215"
4,S1-102811957,"S2-478959098,S2-553508714,S2-625774905,S3-7280..."
5,S1-18727616,"S2-755677256,S3-187831601,S3-641489370,S3-4762..."
6,S1-318373630,"S2-660036492,S3-804600254"
7,S1-86989137,"S3-274817120,S3-312496301"
8,S1-29845983,"S2-648035184,S3-588502663"
9,S1-789009573,"S2-383871912,S3-74481402,S3-576451439"


In [12]:
source_names = [
    "train_source1",
    "train_source2",
    "train_source3",
    "test_source1",
    "test_source2",
    "test_source3",
]

for name in source_names:
    path = FILES[name]

    print("\n" + "=" * 70)
    print(name)
    print("=" * 70)

    query = f"""
    SELECT
        COUNT(*) AS total_rows,

        COUNT(*) - COUNT(entity_id) AS missing_entity_id,
        COUNT(*) - COUNT(business_name) AS missing_business_name,
        COUNT(*) - COUNT(business_address) AS missing_business_address,
        COUNT(*) - COUNT(country) AS missing_country,

        SUM(
            CASE
                WHEN business_name IS NOT NULL
                     AND TRIM(business_name) = ''
                THEN 1 ELSE 0
            END
        ) AS blank_business_name,

        SUM(
            CASE
                WHEN business_address IS NOT NULL
                     AND TRIM(business_address) = ''
                THEN 1 ELSE 0
            END
        ) AS blank_business_address

    FROM read_csv_auto(
        '{path.as_posix()}',
        delim='\t'
    )
    """

    display(con.execute(query).fetchdf())
    


train_source1


,total_rows,missing_entity_id,missing_business_name,missing_business_address,missing_country,blank_business_name,blank_business_address
0,2206821,0,0,0,0,0.0,0.0



train_source2


,total_rows,missing_entity_id,missing_business_name,missing_business_address,missing_country,blank_business_name,blank_business_address
0,5034616,0,0,168967,0,0.0,0.0



train_source3


...

,total_rows,missing_entity_id,missing_business_name,missing_business_address,missing_country,blank_business_name,blank_business_address
0,1732544,0,0,0,0,0.0,0.0



test_source2


,total_rows,missing_entity_id,missing_business_name,missing_business_address,missing_country,blank_business_name,blank_business_address
0,4887273,0,0,129408,0,0.0,0.0



test_source3


,total_rows,missing_entity_id,missing_business_name,missing_business_address,missing_country,blank_business_name,blank_business_address
0,5082316,0,0,136098,0,0.0,0.0


In [13]:
for name in source_names:
    path = FILES[name]

    print("\n" + "=" * 70)
    print(name)
    print("=" * 70)

    query = f"""
    SELECT
        COALESCE(NULLIF(TRIM(country), ''), '<MISSING>') AS country,
        COUNT(*) AS count
    FROM read_csv_auto(
        '{path.as_posix()}',
        delim='\t'
    )
    GROUP BY 1
    ORDER BY count DESC
    """

    display(con.execute(query).fetchdf())


train_source1


,country,count
0,US,1323633
1,India,883188



train_source2


,country,count
0,US,3016817
1,India,2017799



train_source3


...

,country,count
0,India,809986
1,US,663106
2,France,259452



test_source2


,country,count
0,India,2312565
1,US,1871330
2,France,703378



test_source3


,country,count
0,India,2405000
1,US,1945701
2,France,731615


In [14]:
for name in source_names:
    path = FILES[name]

    query = f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT entity_id) AS unique_ids
    FROM read_csv_auto(
        '{path.as_posix()}',
        delim='\t'
    )
    """

    total_rows, unique_ids = con.execute(query).fetchone()

    print(
        f"{name:20} "
        f"rows={total_rows:>10,} "
        f"unique={unique_ids:>10,} "
        f"duplicates={total_rows - unique_ids:>8,}"
    )

train_source1        rows= 2,206,821 unique= 2,206,821 duplicates=       0
train_source2        rows= 5,034,616 unique= 5,034,616 duplicates=       0
train_source3        rows= 5,285,603 unique= 5,285,603 duplicates=       0
test_source1         rows= 1,732,544 unique= 1,732,544 duplicates=       0
test_source2         rows= 4,887,273 unique= 4,887,273 duplicates=       0
test_source3         rows= 5,082,316 unique= 5,082,316 duplicates=       0


In [16]:
gt_path = FILES["train_ground_truth"]

query = f"""
SELECT
    source1_entity_id,
    matched_entity_ids,
    CASE
        WHEN matched_entity_ids IS NULL
             OR TRIM(matched_entity_ids) = ''
        THEN 0
        ELSE len(string_split(matched_entity_ids, ','))
    END AS match_count
FROM read_csv_auto(
    '{gt_path.as_posix()}',
    delim='\t'
)
"""

gt_df = con.execute(query).fetchdf()

gt_df.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,source1_entity_id,matched_entity_ids,match_count
0,S1-965667,"S2-681193310,S2-743505751,S3-775321672,S3-1129...",5
1,S1-55344266,"S2-249013014,S2-197070651,S3-478195123,S3-3843...",4
2,S1-343815751,"S2-790675320,S2-479876582,S3-878454467",3
3,S1-656753428,"S2-153058913,S2-24659151,S3-679606215",3
4,S1-102811957,"S2-478959098,S2-553508714,S2-625774905,S3-7280...",6


In [17]:
match_distribution = (
    gt_df["match_count"]
    .value_counts()
    .sort_index()
    .rename_axis("match_count")
    .reset_index(name="s1_entities")
)

match_distribution


,match_count,s1_entities
0,0,123247
1,1,119157
2,2,375212
3,3,530841
4,4,484115
5,5,321957
6,6,164868
7,7,63968
8,8,18680
9,9,4205


In [18]:
total_s1 = len(gt_df)
singletons = (gt_df["match_count"] == 0).sum()

print("Total S1:", total_s1)
print("Singletons:", singletons)
print("Singleton rate:", round(singletons / total_s1 * 100, 2), "%")
print("Maximum matches for one S1:", gt_df["match_count"].max())

Total S1: 2206821
Singletons: 123247
Singleton rate: 5.58 %
Maximum matches for one S1: 11


In [19]:
matches = (
    gt_df.loc[
        gt_df["match_count"] > 0,
        ["source1_entity_id", "matched_entity_ids"]
    ]
    .assign(
        matched_entity_id=lambda x: x["matched_entity_ids"].str.split(",")
    )
    .explode("matched_entity_id")
)

matches["matched_entity_id"] = matches["matched_entity_id"].str.strip()

reuse = (
    matches.groupby("matched_entity_id")["source1_entity_id"]
    .nunique()
    .sort_values(ascending=False)
)

print(
    "Maximum number of S1 entities sharing one S2/S3 ID:",
    reuse.max()
)

print("\nIDs appearing under multiple S1 entities:")
display(reuse[reuse > 1].head(20))

Maximum number of S1 entities sharing one S2/S3 ID: 1

IDs appearing under multiple S1 entities:


Series([], Name: source1_entity_id, dtype: int64)

In [21]:
s1_path = FILES["train_source1"]
s2_path = FILES["train_source2"]
s3_path = FILES["train_source3"]

train_s1 = con.execute(
    f"""
    SELECT entity_id, country
    FROM read_csv_auto('{s1_path.as_posix()}', delim='\t')
    """
).fetchdf()

train_s2 = con.execute(
    f"""
    SELECT entity_id, country
    FROM read_csv_auto('{s2_path.as_posix()}', delim='\t')
    """
).fetchdf()

train_s3 = con.execute(
    f"""
    SELECT entity_id, country
    FROM read_csv_auto('{s3_path.as_posix()}', delim='\t')
    """
).fetchdf()

targets = pd.concat([train_s2, train_s3], ignore_index=True)
targets = targets.rename(columns={"entity_id": "matched_entity_id"})

pair_countries = (
    matches
    .merge(
        train_s1.rename(columns={"entity_id": "source1_entity_id"}),
        on="source1_entity_id",
        how="left"
    )
    .merge(
        targets,
        on="matched_entity_id",
        how="left"
    )
)

pair_countries["same_country"] = (
    pair_countries["country_x"] == pair_countries["country_y"]
)

print(
    "True pairs with same country:",
    round(pair_countries["same_country"].mean() * 100, 4),
    "%"
)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

True pairs with same country: 100.0 %


In [22]:
pd.crosstab(
    pair_countries["country_x"],
    pair_countries["country_y"],
    margins=True
)

country_y,India,US,All
country_x,,,
India,3059843,0,3059843
US,0,4578522,4578522
All,3059843,4578522,7638365


In [23]:
for name in [
    "train_source1",
    "train_source2",
    "train_source3",
    "test_source1",
    "test_source2",
    "test_source3",
]:
    path = FILES[name]

    query = f"""
    SELECT
        '[' || country || ']' AS raw_country,
        COUNT(*) AS count
    FROM read_csv_auto(
        '{path.as_posix()}',
        delim='\t'
    )
    GROUP BY country
    ORDER BY count DESC
    """

    print("\n" + "=" * 60)
    print(name)
    print("=" * 60)

    display(con.execute(query).fetchdf())


train_source1


,raw_country,count
0,[US],1323633
1,[India],883188



train_source2


,raw_country,count
0,[US],3016817
1,[India],2017799



train_source3


...

,raw_country,count
0,[India],809986
1,[US],663106
2,[France],259452



test_source2


,raw_country,count
0,[India],2312565
1,[US],1871330
2,[France],703378



test_source3


,raw_country,count
0,[India],2405000
1,[US],1945701
2,[France],731615


In [24]:
# Sample S1 entities that have at least one true match
sample_query = f"""
SELECT
    source1_entity_id,
    matched_entity_ids
FROM read_csv_auto(
    '{FILES["train_ground_truth"].as_posix()}',
    delim='\t'
)
WHERE matched_entity_ids IS NOT NULL
  AND TRIM(matched_entity_ids) <> ''
ORDER BY random()
LIMIT 20000
"""

sample_gt = con.execute(sample_query).fetchdf()

print("Sampled S1 entities:", len(sample_gt))

Sampled S1 entities: 20000


In [25]:
sample_gt["matched_entity_id"] = sample_gt["matched_entity_ids"].str.split(",")

sample_pairs = (
    sample_gt[
        ["source1_entity_id", "matched_entity_id"]
    ]
    .explode("matched_entity_id")
)

sample_pairs["matched_entity_id"] = (
    sample_pairs["matched_entity_id"]
    .str.strip()
)

print("Sampled true pairs:", len(sample_pairs))
display(sample_pairs.head())

Sampled true pairs: 73303


,source1_entity_id,matched_entity_id
0,S1-224007531,S2-337390616
0,S1-224007531,S2-114035908
0,S1-224007531,S2-591152339
0,S1-224007531,S3-847919573
1,S1-438920908,S2-849897645


In [29]:
query = f"""
WITH sampled_s1 AS (
    SELECT
        source1_entity_id,
        matched_entity_ids
    FROM read_csv_auto(
        '{FILES["train_ground_truth"].as_posix()}',
        delim='\t'
    )
    WHERE matched_entity_ids IS NOT NULL
      AND TRIM(matched_entity_ids) <> ''
    ORDER BY random()
    LIMIT 20000
),

pairs AS (
    SELECT
        source1_entity_id,
        TRIM(t.matched_entity_id) AS matched_entity_id
    FROM sampled_s1
    CROSS JOIN UNNEST(
        string_split(matched_entity_ids, ',')
    ) AS t(matched_entity_id)
),

targets AS (
    SELECT
        entity_id,
        business_name,
        business_address,
        country
    FROM read_csv_auto(
        '{FILES["train_source2"].as_posix()}',
        delim='\t'
    )

    UNION ALL

    SELECT
        entity_id,
        business_name,
        business_address,
        country
    FROM read_csv_auto(
        '{FILES["train_source3"].as_posix()}',
        delim='\t'
    )
),

joined AS (
    SELECT
        p.source1_entity_id,
        p.matched_entity_id,

        s1.business_name AS s1_name,
        t.business_name AS target_name,

        s1.business_address AS s1_address,
        t.business_address AS target_address,

        s1.country AS s1_country,
        t.country AS target_country

    FROM pairs p

    JOIN read_csv_auto(
        '{FILES["train_source1"].as_posix()}',
        delim='\t'
    ) s1
        ON p.source1_entity_id = s1.entity_id

    JOIN targets t
        ON p.matched_entity_id = t.entity_id
)

SELECT
    COUNT(*) AS true_pairs,

    ROUND(
        100.0 * AVG(
            CASE
                WHEN s1_name = target_name THEN 1
                ELSE 0
            END
        ),
        2
    ) AS raw_name_exact_pct,

    ROUND(
        100.0 * AVG(
            CASE
                WHEN lower(trim(s1_name))
                   = lower(trim(target_name))
                THEN 1
                ELSE 0
            END
        ),
        2
    ) AS normalized_name_exact_pct,

    ROUND(
        100.0 * AVG(
            CASE
                WHEN s1_address IS NOT NULL
                 AND target_address IS NOT NULL
                 AND s1_address = target_address
                THEN 1
                ELSE 0
            END
        ),
        2
    ) AS raw_address_exact_pct,

    ROUND(
        100.0 * AVG(
            CASE
                WHEN s1_address IS NOT NULL
                 AND target_address IS NOT NULL
                 AND lower(trim(s1_address))
                     = lower(trim(target_address))
                THEN 1
                ELSE 0
            END
        ),
        2
    ) AS normalized_address_exact_pct,

    ROUND(
        100.0 * AVG(
            CASE
                WHEN target_address IS NULL THEN 1
                ELSE 0
            END
        ),
        2
    ) AS target_address_missing_pct,

    ROUND(
        100.0 * AVG(
            CASE
                WHEN s1_country = target_country THEN 1
                ELSE 0
            END
        ),
        2
    ) AS same_country_pct

FROM joined
"""

In [30]:
signal_summary = con.execute(query).fetchdf()
display(signal_summary)


,true_pairs,raw_name_exact_pct,normalized_name_exact_pct,raw_address_exact_pct,normalized_address_exact_pct,target_address_missing_pct,same_country_pct
0,73494,4.6,10.72,2.2,7.25,4.41,100.0


# Phase 2 — Scorer & Validation

In [31]:
from pathlib import Path

train_path = FILES["train_ground_truth"].as_posix()

train_out = Path("phase2_train_gt.parquet").resolve()
valid_out = Path("phase2_valid_gt.parquet").resolve()

con.execute(f"""
COPY (
    SELECT *
    FROM read_csv_auto(
        '{train_path}',
        delim='\\t'
    )
    WHERE hash(source1_entity_id) % 5 <> 0
)
TO '{train_out.as_posix()}'
(FORMAT PARQUET)
""")

con.execute(f"""
COPY (
    SELECT *
    FROM read_csv_auto(
        '{train_path}',
        delim='\\t'
    )
    WHERE hash(source1_entity_id) % 5 = 0
)
TO '{valid_out.as_posix()}'
(FORMAT PARQUET)
""")

print("Training split:", train_out)
print("Validation split:", valid_out)

Training split: D:\ML Challenge\6ab10eb3b23ba_student_resource\student_resource\notebooks\phase2_train_gt.parquet
Validation split: D:\ML Challenge\6ab10eb3b23ba_student_resource\student_resource\notebooks\phase2_valid_gt.parquet


In [32]:
split_check = con.execute(f"""
SELECT
    'train' AS split,
    COUNT(*) AS rows
FROM read_parquet('{train_out.as_posix()}')

UNION ALL

SELECT
    'validation' AS split,
    COUNT(*) AS rows
FROM read_parquet('{valid_out.as_posix()}')
""").fetchdf()

display(split_check)

,split,rows
0,train,1765727
1,validation,441094


In [33]:
train_data = con.execute(f"""
    SELECT *
    FROM read_parquet('{train_out.as_posix()}')
    LIMIT 5
""").fetchdf()

display(train_data)
print("Columns:", train_data.columns.tolist())

,source1_entity_id,matched_entity_ids
0,S1-965667,"S2-681193310,S2-743505751,S3-775321672,S3-1129..."
1,S1-343815751,"S2-790675320,S2-479876582,S3-878454467"
2,S1-656753428,"S2-153058913,S2-24659151,S3-679606215"
3,S1-318373630,"S2-660036492,S3-804600254"
4,S1-86989137,"S3-274817120,S3-312496301"


Columns: ['source1_entity_id', 'matched_entity_ids']


In [34]:
for name, path in [
    ("Source 1", s1_path),
    ("Source 2", s2_path),
    ("Source 3", s3_path)
]:
    print(f"\n{'='*50}")
    print(name)
    print("="*50)

    schema = con.execute(f"""
        DESCRIBE
        SELECT *
        FROM read_csv_auto('{path.as_posix()}', delim='\t')
    """).fetchdf()

    display(schema)


Source 1


,column_name,column_type,null,key,default,extra
0,entity_id,VARCHAR,YES,None,None,None
1,business_name,VARCHAR,YES,None,None,None
2,business_address,VARCHAR,YES,None,None,None
3,country,VARCHAR,YES,None,None,None



Source 2


,column_name,column_type,null,key,default,extra
0,entity_id,VARCHAR,YES,None,None,None
1,business_name,VARCHAR,YES,None,None,None
2,business_address,VARCHAR,YES,None,None,None
3,country,VARCHAR,YES,None,None,None



Source 3


,column_name,column_type,null,key,default,extra
0,entity_id,VARCHAR,YES,None,None,None
1,business_name,VARCHAR,YES,None,None,None
2,business_address,VARCHAR,YES,None,None,None
3,country,VARCHAR,YES,None,None,None


In [35]:
for name, path in [
    ("Source 1", s1_path),
    ("Source 2", s2_path),
    ("Source 3", s3_path)
]:
    print(f"\n{'='*50}")
    print(name)
    print("="*50)

    data = con.execute(f"""
        SELECT *
        FROM read_csv_auto('{path.as_posix()}', delim='\t')
        LIMIT 5
    """).fetchdf()

    display(data)


Source 1


,entity_id,business_name,business_address,country
0,S1-925783039,Orelee's Barbershop,"1795 Westchester Drive, High Point, NC",US
1,S1-773889195,Prime Money,"17560 Ellis Road, Tahlequah, OK",US
2,S1-377745466,B+ Retail Inc,"1712 Montebello Avenue, Phoenix, AZ",US
3,S1-133037285,Christ Chapel,"2100 Cameron Drive, Unit APARTMENT G, Dundalk, MD",US
4,S1-755362802,Prabhav Business Center,"797, Lake Town Block A, Kolkata, Howrah, West ...",India



Source 2


,entity_id,business_name,business_address,country
0,S2-166376419,राम मार्केटिंग प्राइवेट लिमिटेड,"KH NO. -570/13, NEW DELHI, WEST DELHI, Delhi",India
1,S2-764573417,-- Holloway Peak Inc Seafood,"105 ELM ST, MORGANTON, NC",US
2,S2-639257739,आदित्य प्रॉपर्टीज एलएलपी,"G-3/571, GULMOHAR COLONY, BHOPAL, Madhya Pradesh",India
3,S2-163963287,Summit Inc,"GREENSBORO, NC, 19 1/2 STARDUST TRAIL",US
4,S2-49942811,Delta Tetlecommunication Inc,"914 PIERPONT AVE, CLEVELAND, OH",US



Source 3


,entity_id,business_name,business_address,country
0,S3-202863386,wilfordhancock.com,"Mack Rd, Haltom City, Texas",US
1,S3-859268022,International South Consultants Private Ltd,NaN,India
2,S3-22467283,LLC Moncada Léarning Center,"5780 Fawn Ct, Fort Worth, Texas",US
3,S3-671162755,Moyna's Coffee,"1 Ivanhoe Ave, PO Box 6009, Cincinnati, Ohio",US
4,S3-960981775,Pvt. EFS Print Ventures Ltd.,"Door No 183, 41St Cross, 22Nd Main 9Th Block J...",India


In [54]:
from typing import Iterable


def parse_ids(value):
    """
    Convert a comma-separated ID string into a set of IDs.
    Empty/NULL values become an empty set.
    """
    if value is None or pd.isna(value):
        return set()

    value = str(value).strip()

    if not value:
        return set()

    return {
        item.strip()
        for item in value.split(",")
        if item.strip()
    }


def f05_score(predicted, actual):
    """
    Compute F0.5 for one Source 1 entity.
    """
    predicted = set(predicted)
    actual = set(actual)

    if not predicted and not actual:
        return 1.0

    if not predicted and actual:
        return 0.0

    if predicted and not actual:
        return 0.0

    true_positive = len(predicted & actual)

    precision = true_positive / len(predicted)
    recall = true_positive / len(actual)

    if precision == 0 and recall == 0:
        return 0.0

    beta = 0.5

    return (
        (1 + beta**2) * precision * recall
        / (beta**2 * precision + recall)
    )


def macro_f05(pred_df, gt_df):
    """
    Compute macro F0.5 across all Source 1 entities.
    
    pred_df columns:
        source1_entity_id
        matched_entity_ids
        
    gt_df columns:
        source1_entity_id
        matched_entity_ids
    """
    pred_map = {
        row["source1_entity_id"]: parse_ids(row["matched_entity_ids"])
        for _, row in pred_df.iterrows()
    }

    gt_map = {
        row["source1_entity_id"]: parse_ids(row["matched_entity_ids"])
        for _, row in gt_df.iterrows()
    }

    scores = []

    for s1_id, actual in gt_map.items():
        predicted = pred_map.get(s1_id, set())
        scores.append(f05_score(predicted, actual))

    return np.mean(scores)

In [55]:
test_cases = [
    # perfect singleton
    (set(), set()),

    # missed real match
    (set(), {"S2-1"}),

    # false positive on singleton
    ({"S2-1"}, set()),

    # perfect one-match prediction
    ({"S2-1"}, {"S2-1"}),

    # partially correct multi-match prediction
    ({"S2-1"}, {"S2-1", "S2-2"}),

    # perfect multi-match prediction
    (
        {"S2-1", "S2-2", "S3-1"},
        {"S2-1", "S2-2", "S3-1"}
    ),

    # extra false positive
    (
        {"S2-1", "S2-2", "S2-3"},
        {"S2-1", "S2-2"}
    ),
]

for i, (pred, actual) in enumerate(test_cases, start=1):
    print(
        f"Case {i}:",
        round(f05_score(pred, actual), 4)
    )

Case 1: 1.0
Case 2: 0.0
Case 3: 0.0
Case 4: 1.0
Case 5: 0.8333
Case 6: 1.0
Case 7: 0.7143


In [56]:
valid_gt = con.execute(f"""
    SELECT *
    FROM read_parquet('{valid_out.as_posix()}')
""").fetchdf()

train_gt = con.execute(f"""
    SELECT *
    FROM read_parquet('{train_out.as_posix()}')
""").fetchdf()

print("Training S1 entities:", len(train_gt))
print("Validation S1 entities:", len(valid_gt))

Training S1 entities: 1765727
Validation S1 entities: 441094


In [57]:
empty_predictions = valid_gt[[
    "source1_entity_id"
]].copy()

empty_predictions["matched_entity_ids"] = ""

baseline_f05 = macro_f05(
    empty_predictions,
    valid_gt
)

print("Empty-prediction baseline macro F0.5:", round(baseline_f05, 6))

Empty-prediction baseline macro F0.5: 0.055811


In [58]:
from pathlib import Path

results_path = ROOT / "experiments" / "results.csv"

baseline_row = pd.DataFrame([{
    "experiment": "empty_prediction_baseline",
    "macro_f05": baseline_f05,
    "notes": "Predict empty match set for every validation S1"
}])

baseline_row.to_csv(
    results_path,
    index=False
)

display(baseline_row)

,experiment,macro_f05,notes
0,empty_prediction_baseline,0.055811,Predict empty match set for every validation S1


In [59]:
from src.scorer import f05_score

def test_scorer_cases():
    cases = [
        (set(), set(), 1.0),
        (set(), {"S2-1"}, 0.0),
        ({"S2-1"}, set(), 0.0),
        ({"S2-1"}, {"S2-1"}, 1.0),
        ({"S2-1"}, {"S2-1", "S2-2"}, 0.8333333333),
        (
            {"S2-1", "S2-2", "S3-1"},
            {"S2-1", "S2-2", "S3-1"},
            1.0
        ),
        (
            {"S2-1", "S2-2", "S2-3"},
            {"S2-1", "S2-2"},
            0.7142857143
        ),
    ]

    for predicted, actual, expected in cases:
        assert abs(f05_score(predicted, actual) - expected) < 1e-6

ModuleNotFoundError: No module named 'src'

In [4]:
from src.normalize import (
    normalize_name,
    normalize_address,
    normalize_country,
)

examples = [
    "ABC Pvt. Ltd.",
    "  ABC   Pvt. Ltd.  ",
    "राम मार्केटिंग प्राइवेट लिमिटेड",
    "LLC Moncada Léarning Center",
    None,
]

for x in examples:
    print("RAW :", repr(x))
    print("NORM:", repr(normalize_name(x)))
    print()

RAW : 'ABC Pvt. Ltd.'
NORM: 'abc pvt. ltd.'

RAW : '  ABC   Pvt. Ltd.  '
NORM: 'abc pvt. ltd.'

RAW : 'राम मार्केटिंग प्राइवेट लिमिटेड'
NORM: 'राम मार्केटिंग प्राइवेट लिमिटेड'

RAW : 'LLC Moncada Léarning Center'
NORM: 'llc moncada léarning center'

RAW : None
NORM: None



In [5]:
sample_gt = con.execute(f"""
    SELECT
        source1_entity_id,
        matched_entity_ids
    FROM read_csv_auto(
        '{FILES["train_ground_truth"].as_posix()}',
        delim='\t'
    )
    WHERE matched_entity_ids IS NOT NULL
      AND TRIM(matched_entity_ids) <> ''
    ORDER BY random()
    LIMIT 20000
""").fetchdf()

sample_gt["matched_entity_id"] = (
    sample_gt["matched_entity_ids"].str.split(",")
)

sample_pairs = (
    sample_gt[
        ["source1_entity_id", "matched_entity_id"]
    ]
    .explode("matched_entity_id")
)

sample_pairs["matched_entity_id"] = (
    sample_pairs["matched_entity_id"].str.strip()
)

print("S1 entities:", len(sample_gt))
print("True pairs:", len(sample_pairs))

NameError: name 'con' is not defined